## Import Libraries

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import pickle

from sklearn.pipeline import Pipeline
from sklearn.model_selection import (GridSearchCV,StratifiedKFold,cross_validate)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

## Model Pipelines
1. Logistic Regression
2. Random Forest
3. Gradient Boosting

In [7]:
%run /Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/StudentPerformance/notebooks/02-data-preprocessing.ipynb


Training Samples: 8000
Testing Samples: 2000

Numerical Columns:
['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score', 'exam_score']

Categorical Columns:
[]


In [8]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=42))
    ])
}

## Grid search parameters

In [9]:
param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100]},

    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_split": [2, 5, 10]},

    "Gradient Boosting": {
        "classifier__n_estimators": [100, 200],
        "classifier__learning_rate": [0.01, 0.05, 0.1],
        "classifier__max_depth": [3, 5]}
}

## Crosss validation setup

In [10]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)

In [11]:
metrics = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted"}

## Hyperparameter Tuning

In [18]:
results = []

for name, pipeline in models.items():

    grid = GridSearchCV(estimator=pipeline,
        param_grid=param_grids[name],
        scoring="f1_weighted",
        cv=cv,
        n_jobs=-1,
        verbose=1)

    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_

    # Cross-validation evaluation
    scores = cross_validate(best_model,
        X_train, y_train,
        cv=cv,
        scoring=metrics,
        n_jobs=-1)


    results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Score": scores["test_f1"].mean()})

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [19]:
# Display results
for result in results:
    print(result)

{'Model': 'Logistic Regression', 'Accuracy': np.float64(0.999375), 'Precision': np.float64(0.9993773602773013), 'Recall': np.float64(0.999375), 'F1 Score': np.float64(0.9993754751358319)}
{'Model': 'Random Forest', 'Accuracy': np.float64(1.0), 'Precision': np.float64(1.0), 'Recall': np.float64(1.0), 'F1 Score': np.float64(1.0)}
{'Model': 'Gradient Boosting', 'Accuracy': np.float64(1.0), 'Precision': np.float64(1.0), 'Recall': np.float64(1.0), 'F1 Score': np.float64(1.0)}


In [20]:
results_df = pd.DataFrame(results)
print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.999375   0.999377  0.999375  0.999375
1        Random Forest  1.000000   1.000000  1.000000  1.000000
2    Gradient Boosting  1.000000   1.000000  1.000000  1.000000


## Cross validation evaluation

In [15]:
cv_scores = cross_validate(
        tuned_model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
